In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into memory (store full tables for subsequent steps)
metric_df = pd.read_csv("metric.csv", low_memory=False)
trace_df = pd.read_csv("trace.csv", low_memory=False)
log_df = pd.read_csv("log.csv", low_memory=False)
error_df = pd.read_csv("error_logs.csv", low_memory=False)

# Helper: convert epoch-second series to Asia/Shanghai timezone-aware min/max strings
tz = pytz.timezone('Asia/Shanghai')
def ts_min_max(series):
    if series.dropna().shape[0] == 0:
        return pd.NaT, pd.NaT
    # interpret timestamps as UTC epoch seconds, then convert to Asia/Shanghai
    s = pd.to_datetime(series.astype(int), unit='s', utc=True)
    s_local = s.dt.tz_convert(tz)
    return s_local.min(), s_local.max()

# 1) For each file: row count, min timestamp, max timestamp
files = []
for name, df in [("metric.csv", metric_df), ("trace.csv", trace_df), ("log.csv", log_df), ("error_logs.csv", error_df)]:
    rows = df.shape[0]
    mn, mx = (pd.NaT, pd.NaT)
    if "timestamp" in df.columns and rows>0:
        mn, mx = ts_min_max(df["timestamp"])
    files.append({"file": name, "rows": rows, "min_ts_Asia/Shanghai": mn, "max_ts_Asia/Shanghai": mx})
files_summary = pd.DataFrame(files)

# 2) For each file: sorted unique cmdb_id values (up to 20)
def top_cmdbs(df):
    if "cmdb_id" not in df.columns:
        return pd.Series([], dtype=object)
    uniques = sorted(df["cmdb_id"].dropna().astype(str).unique())
    return pd.Series(uniques[:20], name="cmdb_id")
metric_cmdbs = top_cmdbs(metric_df)
trace_cmdbs = top_cmdbs(trace_df)
log_cmdbs = top_cmdbs(log_df)
error_cmdbs = top_cmdbs(error_df)

# 3) metric.csv specifics: up to 20 unique kpi_name values (sorted) and up to 20 unique (cmdb_id,kpi_name) pairs
if "kpi_name" in metric_df.columns:
    metric_kpis = pd.Series(sorted(metric_df["kpi_name"].dropna().astype(str).unique())[:20], name="kpi_name")
else:
    metric_kpis = pd.Series([], dtype=object, name="kpi_name")

if {"cmdb_id","kpi_name"}.issubset(metric_df.columns):
    metric_pairs = (metric_df[["cmdb_id","kpi_name"]]
                    .dropna()
                    .astype(str)
                    .drop_duplicates()
                    .sort_values(["cmdb_id","kpi_name"])
                    .reset_index(drop=True)
                    .head(20))
else:
    metric_pairs = pd.DataFrame(columns=["cmdb_id","kpi_name"])

# 4) trace.csv specifics: up to 20 unique trace_name and up to 20 (cmdb_id,trace_name)
if "trace_name" in trace_df.columns:
    trace_kpis = pd.Series(sorted(trace_df["trace_name"].dropna().astype(str).unique())[:20], name="trace_name")
else:
    trace_kpis = pd.Series([], dtype=object, name="trace_name")

if {"cmdb_id","trace_name"}.issubset(trace_df.columns):
    trace_pairs = (trace_df[["cmdb_id","trace_name"]]
                   .dropna()
                   .astype(str)
                   .drop_duplicates()
                   .sort_values(["cmdb_id","trace_name"])
                   .reset_index(drop=True)
                   .head(20))
else:
    trace_pairs = pd.DataFrame(columns=["cmdb_id","trace_name"])

# 5) log.csv specifics: up to 20 unique log_name and up to 20 (cmdb_id,log_name)
if "log_name" in log_df.columns:
    log_kpis = pd.Series(sorted(log_df["log_name"].dropna().astype(str).unique())[:20], name="log_name")
else:
    log_kpis = pd.Series([], dtype=object, name="log_name")

if {"cmdb_id","log_name"}.issubset(log_df.columns):
    log_pairs = (log_df[["cmdb_id","log_name"]]
                 .dropna()
                 .astype(str)
                 .drop_duplicates()
                 .sort_values(["cmdb_id","log_name"])
                 .reset_index(drop=True)
                 .head(20))
else:
    log_pairs = pd.DataFrame(columns=["cmdb_id","log_name"])

# 6) error_logs.csv specifics: up to 20 distinct messages (first 200 chars) and up to 20 unique cmdb_id values (sorted)
if "message" in error_df.columns:
    messages = error_df["message"].dropna().astype(str).unique()
    # preserve order of appearance but limit to 20, and truncate to 200 chars
    seen = []
    previews = []
    for msg in messages:
        if len(seen) >= 20:
            break
        truncated = msg if len(msg) <= 200 else msg[:200]
        previews.append(truncated)
        seen.append(truncated)
    error_messages = pd.Series(previews, name="message_preview").reset_index(drop=True)
else:
    error_messages = pd.Series([], dtype=object, name="message_preview")

# Compact outputs (limited lists / top 20)
# files_summary, *_cmdbs are small; *_kpis and *_pairs are limited to 20; error_messages limited to 20.
files_summary, metric_cmdbs, metric_kpis, metric_pairs, trace_cmdbs, trace_kpis, trace_pairs, log_cmdbs, log_kpis, log_pairs, error_messages, error_cmdbs

```
Out[1]:
```
from datetime import datetime
# Build a concise plain-English summary using variables from the execution context
metric_rows = files_summary.loc[files_summary['file']=='metric.csv', 'rows'].values[0]
metric_min = files_summary.loc[files_summary['file']=='metric.csv', 'min_ts_Asia/Shanghai'].values[0]
metric_max = files_summary.loc[files_summary['file']=='metric.csv', 'max_ts_Asia/Shanghai'].values[0]

trace_rows = files_summary.loc[files_summary['file']=='trace.csv', 'rows'].values[0]
trace_min = files_summary.loc[files_summary['file']=='trace.csv', 'min_ts_Asia/Shanghai'].values[0]
trace_max = files_summary.loc[files_summary['file']=='trace.csv', 'max_ts_Asia/Shanghai'].values[0]

log_rows = files_summary.loc[files_summary['file']=='log.csv', 'rows'].values[0]
log_min = files_summary.loc[files_summary['file']=='log.csv', 'min_ts_Asia/Shanghai'].values[0]
log_max = files_summary.loc[files_summary['file']=='log.csv', 'max_ts_Asia/Shanghai'].values[0]

err_rows = files_summary.loc[files_summary['file']=='error_logs.csv', 'rows'].values[0]
err_min = files_summary.loc[files_summary['file']=='error_logs.csv', 'min_ts_Asia/Shanghai'].values[0]
err_max = files_summary.loc[files_summary['file']=='error_logs.csv', 'max_ts_Asia/Shanghai'].values[0]

# Prepare compact lists (already limited to 20 in prior step)
metric_cmdb_list = metric_cmdbs.tolist()
metric_kpi_list = metric_kpis.tolist()
metric_pair_preview = metric_pairs.head(5).to_dict(orient='records')  # show a few examples

trace_cmdb_list = trace_cmdbs.tolist()
trace_kpi_list = trace_kpis.tolist()
trace_pair_preview = trace_pairs.head(5).to_dict(orient='records')

log_cmdb_list = log_cmdbs.tolist()
log_kpi_list = log_kpis.tolist()
log_pair_preview = log_pairs.head(6).to_dict(orient='records')

error_messages_list = error_messages.tolist()
error_cmdb_list = error_cmdbs.tolist()

summary = (
    "Telemetry summary (compact):\n\n"
    f"1) metric.csv: {metric_rows} rows; time range {metric_min} to {metric_max} (Asia/Shanghai).\n"
    f"   - Up to 20 cmdb_ids (sample): {', '.join(metric_cmdb_list)}\n"
    f"   - Up to 20 kpi_name samples: {', '.join(metric_kpi_list[:8])}"  # keep this short
    f"\n   - Example (cmdb_id, kpi_name) pairs (first 5): {metric_pair_preview}\n\n"
    f"2) trace.csv: {trace_rows} rows; time range {trace_min} to {trace_max} (Asia/Shanghai).\n"
    f"   - cmdb_ids: {', '.join(trace_cmdb_list)}\n"
    f"   - trace_name samples: {', '.join(trace_kpi_list[:8])}\n"
    f"\n   - Example (cmdb_id, trace_name) pairs (first 5): {trace_pair_preview}\n\n"
    f"3) log.csv: {log_rows} rows; time range {log_min} to {log_max} (Asia/Shanghai).\n"
    f"   - cmdb_ids: {', '.join(log_cmdb_list)}\n"
    f"   - log_name values: {', '.join(log_kpi_list)}\n"
    f"\n   - Example (cmdb_id, log_name) pairs: {log_pair_preview}\n\n"
    f"4) error_logs.csv: {err_rows} rows; no timestamp range available.\n"
    f"   - Stored message previews (up to 20): {error_messages_list if error_messages_list else 'NONE'}\n"
    f"   - cmdb_ids seen in error_logs (up to 20): {error_cmdb_list if error_cmdb_list else 'NONE'}\n\n"
    "Notes: metric.csv contains many JVM/OS and container KPIs (examples shown). trace.csv includes both self- and to-/from- traces (per service). log.csv contains error_count/gc_pause_max/row_count. error_logs.csv is empty in this dataset."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows      min_ts_Asia/Shanghai      max_ts_Asia/Shanghai
0      metric.csv  27485 2021-03-09 01:00:00+08:00 2021-03-09 01:30:00+08:00
1       trace.csv   5862 2021-03-09 01:00:00+08:00 2021-03-09 01:29:00+08:00
2         log.csv    399 2021-03-09 01:00:00+08:00 2021-03-09 01:29:00+08:00
3  error_logs.csv      0                       NaT                       NaT, 0              IG01
1              IG02
2              MG01
3              MG02
4           Mysql01
5           Mysql02
6           Redis01
7           Redis02
8      ServiceTest1
9     ServiceTest10
10    ServiceTest11
11     ServiceTest2
12     ServiceTest3
13     ServiceTest4
14     ServiceTest5
15     ServiceTest6
16     ServiceTest7
17     ServiceTest8
18     ServiceTest9
19         Tomcat01
Name: cmdb_id, dtype: object, 0     Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...
1     Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...
2     Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...
3     Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...
4     Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...
5     Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...
6     Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...
7     Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...
8     Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...
9     Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...
10    Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...
11    Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
12    Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
13    Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
14    Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
15    Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
16    Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
17    Container-DOCKER_CONTAINER_b2fc383d2438--bcou-...
18    Container-DOCKER_CONTAINER_b2fc383d2438--bcou-...
19    Container-DOCKER_CONTAINER_b2fc383d2438--bcou-...
Name: kpi_name, dtype: object,    cmdb_id                                         kpi_name
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax
1     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
2     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
3     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
4     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad
5     IG01                  JVM-Runtime_7778_JVM_JVM_Uptime
6     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads
7     IG01                  OSLinux-CPU_CPU-0_SingleCpuUtil
8     IG01                  OSLinux-CPU_CPU-0_SingleCpuidle
9     IG01                  OSLinux-CPU_CPU-2_SingleCpuUtil
10    IG01                  OSLinux-CPU_CPU-2_SingleCpuidle
11    IG01                       OSLinux-CPU_CPU_CPUCpuUtil
12    IG01                          OSLinux-CPU_CPU_CPULoad
13    IG01                       OSLinux-CPU_CPU_CPUSysTime
14    IG01                      OSLinux-CPU_CPU_CPUUserTime
15    IG01                           OSLinux-CPU_CPU_CPUWio
16    IG01                      OSLinux-CPU_CPU_CPUidleutil
17    IG01    OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace
18    IG01          OSLinux-OSLinux_FILESYSTEM_-_FSCapacity
19    IG01  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat03
6     Tomcat04
7     dockerA1
8     dockerA2
9     dockerB1
10    dockerB2
Name: cmdb_id, dtype: object, 0         trace.from_IG01.duration_mean
1          trace.from_IG01.duration_p95
2             trace.from_IG01.row_count
3         trace.from_IG02.duration_mean
4          trace.from_IG02.duration_p95
5             trace.from_IG02.row_count
6         trace.from_MG01.duration_mean
7          trace.from_MG01.duration_p95
8             trace.from_MG01.row_count
9         trace.from_MG02.duration_mean
10         trace.from_MG02.duration_p95
11            trace.from_MG02.row_count
12    trace.from_Tomcat01.duration_mean
13     trace.from_Tomcat01.duration_p95
14        trace.from_Tomcat01.row_count
15    trace.from_Tomcat03.duration_mean
16     trace.from_Tomcat03.duration_p95
17        trace.from_Tomcat03.row_count
18    trace.from_Tomcat04.duration_mean
19     trace.from_Tomcat04.duration_p95
Name: trace_name, dtype: object,    cmdb_id                       trace_name
0     IG01         trace.self.duration_mean
1     IG01          trace.self.duration_p95
2     IG01             trace.self.row_count
3     IG01  trace.to_Tomcat01.duration_mean
4     IG01   trace.to_Tomcat01.duration_p95
5     IG01      trace.to_Tomcat01.row_count
6     IG01  trace.to_Tomcat03.duration_mean
7     IG01   trace.to_Tomcat03.duration_p95
8     IG01      trace.to_Tomcat03.row_count
9     IG01  trace.to_Tomcat04.duration_mean
10    IG01   trace.to_Tomcat04.duration_p95
11    IG01      trace.to_Tomcat04.row_count
12    IG02         trace.self.duration_mean
13    IG02          trace.self.duration_p95
14    IG02             trace.self.row_count
15    IG02  trace.to_Tomcat01.duration_mean
16    IG02   trace.to_Tomcat01.duration_p95
17    IG02      trace.to_Tomcat01.row_count
18    IG02  trace.to_Tomcat03.duration_mean
19    IG02   trace.to_Tomcat03.duration_p95, 0    Tomcat01
1    Tomcat02
2    Tomcat03
3    Tomcat04
4    apache01
5    apache02
Name: cmdb_id, dtype: object, 0     log.error_count
1    log.gc_pause_max
2       log.row_count
Name: log_name, dtype: object,      cmdb_id          log_name
0   Tomcat01   log.error_count
1   Tomcat01  log.gc_pause_max
2   Tomcat01     log.row_count
3   Tomcat02   log.error_count
4   Tomcat02  log.gc_pause_max
5   Tomcat02     log.row_count
6   Tomcat03   log.error_count
7   Tomcat03  log.gc_pause_max
8   Tomcat03     log.row_count
9   Tomcat04   log.error_count
10  Tomcat04  log.gc_pause_max
11  Tomcat04     log.row_count
12  apache01   log.error_count
13  apache01     log.row_count
14  apache02   log.error_count
15  apache02     log.row_count, Series([], Name: message_preview, dtype: object), Series([], Name: cmdb_id, dtype: object))```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df and tz if available from previous cell; otherwise read again (should be in memory)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv", low_memory=False)

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P90 for every (cmdb_id, kpi_name) pair using the full series (global thresholds)
thresholds = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"], dropna=False)["value"]
    .agg(global_P90=lambda x: x.quantile(0.90),
         global_P95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter metric rows to the incident window inclusive between 2021-03-09 01:00:00+08 and 2021-03-09 01:30:00+08
# Create tz-aware local timestamp column for filtering and later reporting
metric_df["ts_local"] = pd.to_datetime(metric_df["timestamp"].astype(int), unit="s", utc=True).dt.tz_convert(tz)

start = pd.Timestamp("2021-03-09 01:00:00", tz=tz)
end = pd.Timestamp("2021-03-09 01:30:00", tz=tz)

window_df = metric_df[(metric_df["ts_local"] >= start) & (metric_df["ts_local"] <= end)].copy()

# 3) For each (cmdb_id, kpi_name) within the filtered window compute required metrics
# Merge global P95 into the windowed data (global thresholds computed BEFORE filtering as required)
window_with_thresh = window_df.merge(thresholds[["cmdb_id", "kpi_name", "global_P95"]], on=["cmdb_id", "kpi_name"], how="left")

# Group and aggregate per pair
def compute_metrics(g):
    total_points = int(g.shape[0])
    max_value = g["value"].max()
    global_p95 = g["global_P95"].iloc[0] if "global_P95" in g.columns else np.nan

    # count_high_anomalies where value >= global_P95 (handle NaN)
    if pd.isna(global_p95):
        count_high = 0
        earliest_high_ts = pd.NaT
        severity = np.nan
    else:
        mask = g["value"] >= global_p95
        count_high = int(mask.sum())
        earliest_high_ts = g.loc[mask, "ts_local"].min() if count_high > 0 else pd.NaT
        if global_p95 == 0:
            severity = np.nan
        else:
            severity = (max_value - global_p95) / global_p95

    return pd.Series({
        "total_points_in_window": total_points,
        "count_high_anomalies": count_high,
        "earliest_high_anomaly_ts_Asia/Shanghai": earliest_high_ts,
        "max_value_in_window": max_value,
        "global_P95": global_p95,
        "severity_ratio": severity
    })

agg = (
    window_with_thresh
    .groupby(["cmdb_id", "kpi_name"], dropna=False)
    .apply(compute_metrics)
    .reset_index()
)

# 4) Return compact table limited to top 20 rows sorted by count_high_anomalies desc, tie-breaker severity_ratio desc.
# Only include pairs that exist in the window. For "high anomalies" focus, we will return rows with count_high_anomalies > 0.
high_anoms = agg[agg["count_high_anomalies"] > 0].copy()

# Sort and select top 20
top20 = high_anoms.sort_values(by=["count_high_anomalies", "severity_ratio"], ascending=[False, False]).head(20)

# Keep columns in requested order
result_table = top20[[
    "cmdb_id",
    "kpi_name",
    "total_points_in_window",
    "count_high_anomalies",
    "earliest_high_anomaly_ts_Asia/Shanghai",
    "max_value_in_window",
    "global_P95",
    "severity_ratio"
]].reset_index(drop=True)

# 5) Scalars
number_of_unique_component_kpis_processed = int(agg.shape[0])   # unique (cmdb_id,kpi_name) pairs present in the window
total_metric_rows_in_window = int(window_df.shape[0])

# If no high anomalies found, prepare a message
if result_table.shape[0] == 0:
    no_high_anomalies_message = ("No high anomalies found: no (cmdb_id, kpi_name) in the incident window "
                                "had value >= their global P95. You may try lower thresholds or check for low-value anomalies.")
else:
    no_high_anomalies_message = ""

# Final compact outputs (stored in variables for inspection)
result_table, number_of_unique_component_kpis_processed, total_metric_rows_in_window, no_high_anomalies_message

```
Out[2]:
```
summary = (
    f"Summary of metric anomaly analysis (incident window 2021-03-09 01:00:00+08 to 2021-03-09 01:30:00+08):\n\n"
    f"- Global P95/P90 thresholds were computed per (cmdb_id, kpi_name) using the full metric series.\n"
    f"- Window filtering produced {total_metric_rows_in_window} metric rows covering {number_of_unique_component_kpis_processed} unique (cmdb_id, kpi_name) pairs.\n"
    f"- High anomalies were defined as points with value >= the pair's global P95. The analysis found high anomalies (some pairs have all window points >= global P95).\n\n"
)

if result_table.shape[0] == 0:
    summary += "No high anomalies found in the incident window (no points >= global P95). You may try lower thresholds or look for low-value anomalies."
else:
    top_n = result_table.shape[0]
    # Provide top example details (first row)
    r0 = result_table.iloc[0]
    summary += (
        f"- Top {top_n} (cmdb_id, kpi_name) pairs sorted by count_high_anomalies then severity_ratio are shown in the stored result_table.\n\n"
        f"Top example (row 0):\n"
        f"  cmdb_id: {r0['cmdb_id']}\n"
        f"  kpi_name: {r0['kpi_name']}\n"
        f"  total_points_in_window: {int(r0['total_points_in_window'])}\n"
        f"  count_high_anomalies: {int(r0['count_high_anomalies'])}\n"
        f"  earliest_high_anomaly_ts_Asia/Shanghai: {r0['earliest_high_anomaly_ts_Asia/Shanghai']}\n"
        f"  max_value_in_window: {r0['max_value_in_window']}\n"
        f"  global_P95: {r0['global_P95']}\n"
        f"  severity_ratio: {r0['severity_ratio']}\n\n"
        "Notes:\n"
        "- Many listed pairs have count_high_anomalies equal to the full window length (31), with severity_ratio 0.0 where max == global_P95.\n"
        "- A notable non-zero severity example is the top row: value 0.40 vs global_P95 0.30 (severity_ratio ~0.33).\n"
        "- Earliest high-anomaly timestamps in the top results are at the window start (2021-03-09 01:00:00+08:00).\n\n"
        "Stored outputs:\n"
        "- result_table: compact top-20 table with requested columns\n"
        "- number_of_unique_component_kpis_processed and total_metric_rows_in_window scalars\n"
        "Use those variables for further inspection or lowering thresholds if needed."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  total_points_in_window  count_high_anomalies earliest_high_anomaly_ts_Asia/Shanghai  max_value_in_window    global_P95  severity_ratio
0       IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                      31                    31              2021-03-09 01:00:00+08:00         4.000000e-01  3.000000e-01        0.333333
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                      31                    31              2021-03-09 01:00:00+08:00         2.040136e+09  2.040136e+09        0.000000
2       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                      31                    31              2021-03-09 01:00:00+08:00         5.787243e+07  5.787243e+07        0.000000
3       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31                    31              2021-03-09 01:00:00+08:00         1.000000e+00  1.000000e+00        0.000000
4       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31                    31              2021-03-09 01:00:00+08:00         3.000000e+00  3.000000e+00        0.000000
5       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                      31                    31              2021-03-09 01:00:00+08:00         2.040136e+09  2.040136e+09        0.000000
6       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                      31                    31              2021-03-09 01:00:00+08:00         5.806302e+07  5.806302e+07        0.000000
7       IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31                    31              2021-03-09 01:00:00+08:00         1.000000e+00  1.000000e+00        0.000000
8       IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31                    31              2021-03-09 01:00:00+08:00         3.000000e+00  3.000000e+00        0.000000
9       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                      31                    31              2021-03-09 01:00:00+08:00         2.040136e+09  2.040136e+09        0.000000
10      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                      31                    31              2021-03-09 01:00:00+08:00         5.069042e+08  5.069042e+08        0.000000
11      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                      31                    31              2021-03-09 01:00:00+08:00         2.040136e+09  2.040136e+09        0.000000
12      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                      31                    31              2021-03-09 01:00:00+08:00         9.471424e+08  9.471424e+08        0.000000
13   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31                    31              2021-03-09 01:00:00+08:00         2.400000e+01  2.400000e+01        0.000000
14   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31                    31              2021-03-09 01:00:00+08:00         4.000000e+00  4.000000e+00        0.000000
15  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31                    31              2021-03-09 01:00:00+08:00         6.420000e+02  6.420000e+02        0.000000
16  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31                    31              2021-03-09 01:00:00+08:00         4.053960e+05  4.053960e+05        0.000000
17  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31                    31              2021-03-09 01:00:00+08:00         1.000000e+02  1.000000e+02        0.000000
18  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31                    31              2021-03-09 01:00:00+08:00         5.000000e+02  5.000000e+02        0.000000
19  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31                    31              2021-03-09 01:00:00+08:00         5.580000e+02  5.580000e+02        0.000000, 1688, 27485, '')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing in-memory variables: metric_df, thresholds, window_df, window_with_thresh, agg
# If any missing, raise to avoid silent mistakes
for var in ["metric_df", "thresholds", "window_df", "window_with_thresh", "agg"]:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel. Ensure prior steps executed.")

tz = pytz.timezone("Asia/Shanghai")

# 1) Select candidate pairs where severity_ratio > 0 OR count_high_anomalies >= 3
candidates = agg[
    (agg["severity_ratio"].fillna(0) > 0) |
    (agg["count_high_anomalies"].fillna(0) >= 3)
].copy()

# Exclude pairs without a valid global_P95 (cannot evaluate anomalies)
candidates = candidates[~candidates["global_P95"].isna()].reset_index(drop=True)

# If no candidates, prepare empty outputs
if candidates.shape[0] == 0:
    segments_df = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","segment_start_ts_Asia/Shanghai","segment_end_ts_Asia/Shanghai",
        "segment_length_in_points","max_value_in_segment","global_P95","severity_ratio"
    ])
    unique_cmdbs = pd.Series([], dtype=object, name="cmdb_id")
else:
    # 2) For each candidate pair, scan incident-window rows ordered by timestamp and extract consecutive anomaly segments
    # Filter window_with_thresh to only candidate pairs
    # Ensure ts_local exists and is timezone-aware (it was created previously)
    cand_keys = candidates[["cmdb_id", "kpi_name"]].drop_duplicates()
    merged_window = window_with_thresh.merge(cand_keys, on=["cmdb_id", "kpi_name"], how="inner")

    # Prepare list to collect segments
    seg_list = []

    # Group by pair
    grp = merged_window.sort_values(["cmdb_id", "kpi_name", "ts_local"]).groupby(["cmdb_id", "kpi_name"], sort=False)
    for (cmdb_id, kpi_name), g in grp:
        # ensure sorted by timestamp
        g = g.sort_values("ts_local").reset_index(drop=True)
        # global_P95 for this group (should be same for all rows)
        global_p95 = g["global_P95"].iloc[0]
        # mask of anomaly points
        mask = g["value"] >= global_p95
        if mask.sum() == 0:
            continue
        # identify runs: use cumulative change in mask to label segments
        run_id = (mask != mask.shift(fill_value=False)).cumsum()
        g = g.assign(run_id=run_id, is_anom=mask)
        # iterate runs where is_anom True
        for run, sub in g.groupby("run_id"):
            if not sub["is_anom"].iloc[0]:
                continue
            seg_start = sub["ts_local"].iloc[0]
            seg_end = sub["ts_local"].iloc[-1]
            seg_len = int(sub.shape[0])
            max_val = float(sub["value"].max())
            # severity based on segment max
            if global_p95 == 0:
                seg_severity = np.nan
            else:
                seg_severity = (max_val - global_p95) / global_p95
            seg_list.append({
                "cmdb_id": cmdb_id,
                "kpi_name": kpi_name,
                "segment_start_ts_Asia/Shanghai": seg_start,
                "segment_end_ts_Asia/Shanghai": seg_end,
                "segment_length_in_points": seg_len,
                "max_value_in_segment": max_val,
                "global_P95": float(global_p95),
                "severity_ratio": float(seg_severity) if not np.isnan(seg_severity) else np.nan
            })

    segments_df = pd.DataFrame(seg_list)

    # 3) Filter out segments with length == 1 unless severity_ratio > 0.5
    if not segments_df.empty:
        cond_keep = (segments_df["segment_length_in_points"] > 1) | (segments_df["severity_ratio"] > 0.5)
        segments_df = segments_df[cond_keep].reset_index(drop=True)

    # 4) Prepare compact outputs: top 20 segments sorted by length desc then severity desc
    if not segments_df.empty:
        segments_df = segments_df.sort_values(by=["segment_length_in_points", "severity_ratio"], ascending=[False, False]).reset_index(drop=True)
        top_segments = segments_df.head(20).copy()
        unique_cmdbs = pd.Series(sorted(top_segments["cmdb_id"].dropna().unique())[:10], name="cmdb_id")
        segments_df = top_segments  # keep compact table in this variable
    else:
        # no segments after filtering
        segments_df = pd.DataFrame(columns=[
            "cmdb_id","kpi_name","segment_start_ts_Asia/Shanghai","segment_end_ts_Asia/Shanghai",
            "segment_length_in_points","max_value_in_segment","global_P95","severity_ratio"
        ])
        unique_cmdbs = pd.Series([], dtype=object, name="cmdb_id")

# Final returned compact variables
segments_df, unique_cmdbs

```
Out[3]:
```
from datetime import datetime

# Build concise plain-English summary using stored variables
count_segments = int(segments_df.shape[0])
unique_cmdbs_list = unique_cmdbs.tolist()

summary = (
    f"Segment extraction summary:\n\n"
    f"- Number of anomaly segments after filtering: {count_segments} (limited to top 20 shown).\n"
    f"- Unique components in these segments (up to 10): {', '.join(unique_cmdbs_list)}.\n\n"
)

if count_segments == 0:
    summary += ("No segments remained after the noise filter (length==1 suppressed unless severity>50%). "
                "You can relax the filtering (allow length==1 or lower threshold) to find more short-lived anomalies.")
else:
    top0 = segments_df.iloc[0]
    summary += (
        f"- Top segment example:\n"
        f"  * cmdb_id: {top0['cmdb_id']}\n"
        f"  * kpi_name: {top0['kpi_name']}\n"
        f"  * segment_start_ts_Asia/Shanghai: {top0['segment_start_ts_Asia/Shanghai']}\n"
        f"  * segment_end_ts_Asia/Shanghai: {top0['segment_end_ts_Asia/Shanghai']}\n"
        f"  * segment_length_in_points: {int(top0['segment_length_in_points'])}\n"
        f"  * max_value_in_segment: {top0['max_value_in_segment']}\n"
        f"  * global_P95: {top0['global_P95']}\n"
        f"  * severity_ratio: {top0['severity_ratio']}\n\n"
        "Observations:\n"
        "- Many segments span the entire incident window (length 31), indicating persistent values >= global P95 for those pairs.\n"
        "- Most segments have severity_ratio == 0.0 (max == global_P95); one notable non-zero example: IG02 zabbix agentd process (severity ~0.33).\n"
        "- If you want more/shorter segments, relax the filter to include single-point segments or lower the threshold below global P95.\n"
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name segment_start_ts_Asia/Shanghai segment_end_ts_Asia/Shanghai  segment_length_in_points  max_value_in_segment    global_P95  severity_ratio
0       IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          4.000000e-01  3.000000e-01        0.333333
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          2.040136e+09  2.040136e+09        0.000000
2       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          5.787243e+07  5.787243e+07        0.000000
3       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          1.000000e+00  1.000000e+00        0.000000
4       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          3.000000e+00  3.000000e+00        0.000000
5       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          2.040136e+09  2.040136e+09        0.000000
6       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          5.806302e+07  5.806302e+07        0.000000
7       IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          1.000000e+00  1.000000e+00        0.000000
8       IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          3.000000e+00  3.000000e+00        0.000000
9       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          2.040136e+09  2.040136e+09        0.000000
10      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          5.069042e+08  5.069042e+08        0.000000
11      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          2.040136e+09  2.040136e+09        0.000000
12      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          9.471424e+08  9.471424e+08        0.000000
13   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          2.400000e+01  2.400000e+01        0.000000
14   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          4.000000e+00  4.000000e+00        0.000000
15  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          6.420000e+02  6.420000e+02        0.000000
16  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          4.053960e+05  4.053960e+05        0.000000
17  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          1.000000e+02  1.000000e+02        0.000000
18  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          5.000000e+02  5.000000e+02        0.000000
19  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...      2021-03-09 01:00:00+08:00    2021-03-09 01:30:00+08:00                        31          5.580000e+02  5.580000e+02        0.000000, 0        IG01
1        IG02
2        MG01
3        MG02
4     Mysql02
5     Redis02
6    Tomcat01
7    Tomcat03
Name: cmdb_id, dtype: object)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv", low_memory=False)

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P90 for each (cmdb_id, trace_name) using the full series (global thresholds)
trace_thresholds = (
    trace_df
    .groupby(["cmdb_id", "trace_name"], dropna=False)["value"]
    .agg(global_P90=lambda x: x.quantile(0.90),
         global_P95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter trace rows to the incident window inclusive (Asia/Shanghai)
# Create tz-aware local timestamp column for filtering/reporting (reuse if exists)
trace_df["ts_local"] = pd.to_datetime(trace_df["timestamp"].astype(int), unit="s", utc=True).dt.tz_convert(tz)

start = pd.Timestamp("2021-03-09 01:00:00", tz=tz)
end = pd.Timestamp("2021-03-09 01:30:00", tz=tz)

trace_window = trace_df[(trace_df["ts_local"] >= start) & (trace_df["ts_local"] <= end)].copy()

# 3) For each (cmdb_id, trace_name) in the filtered window compute requested metrics using global_P95 from full series
# Merge global_P95 into the windowed data (thresholds computed BEFORE filtering)
trace_window = trace_window.merge(trace_thresholds[["cmdb_id", "trace_name", "global_P95"]], on=["cmdb_id", "trace_name"], how="left")

def compute_trace_metrics(g):
    total_points = int(g.shape[0])
    max_value = g["value"].max()
    global_p95 = g["global_P95"].iloc[0] if "global_P95" in g.columns else np.nan

    if pd.isna(global_p95):
        count_high = 0
        earliest_high_ts = pd.NaT
        severity = np.nan
    else:
        mask = g["value"] >= global_p95
        count_high = int(mask.sum())
        earliest_high_ts = g.loc[mask, "ts_local"].min() if count_high > 0 else pd.NaT
        if global_p95 == 0:
            severity = np.nan
        else:
            severity = (max_value - global_p95) / global_p95

    return pd.Series({
        "total_points_in_window": total_points,
        "count_high_anomalies": count_high,
        "earliest_high_anomaly_ts_Asia/Shanghai": earliest_high_ts,
        "max_value_in_window": max_value,
        "global_P95": global_p95,
        "severity_ratio": severity
    })

trace_agg = (
    trace_window
    .groupby(["cmdb_id", "trace_name"], dropna=False)
    .apply(compute_trace_metrics)
    .reset_index()
)

# 4) Prepare compact table: top 20 rows sorted by count_high_anomalies desc, tie-breaker severity_ratio desc
trace_high = trace_agg[trace_agg["count_high_anomalies"] > 0].copy()

if trace_high.shape[0] == 0:
    trace_result_table = pd.DataFrame(columns=[
        "cmdb_id","trace_name","total_points_in_window","count_high_anomalies",
        "earliest_high_anomaly_ts_Asia/Shanghai","max_value_in_window","global_P95","severity_ratio"
    ])
    trace_unique_cmdbs = pd.Series([], dtype=object, name="cmdb_id")
    trace_no_high_anoms_message = ("No high anomalies found in traces at >= global P95 in the incident window. "
                                   "Consider relaxing to >= P90 in the next step.")
else:
    trace_result_table = trace_high.sort_values(by=["count_high_anomalies", "severity_ratio"], ascending=[False, False]).head(20)
    # Keep only requested columns and reset index
    trace_result_table = trace_result_table[[
        "cmdb_id","trace_name","total_points_in_window","count_high_anomalies",
        "earliest_high_anomaly_ts_Asia/Shanghai","max_value_in_window","global_P95","severity_ratio"
    ]].reset_index(drop=True)
    trace_unique_cmdbs = pd.Series(sorted(trace_result_table["cmdb_id"].dropna().unique())[:10], name="cmdb_id")
    trace_no_high_anoms_message = ""

# Final compact outputs (variables for inspection)
trace_result_table, trace_unique_cmdbs, trace_no_high_anoms_message

```
Out[4]:
```
Summary of trace-level anomaly detection (incident window 2021-03-09 01:00–01:30 Asia/Shanghai):

- Overview:
  - High-anomaly traces (value >= per-pair global P95) were found. The returned compact table shows the top 20 trace (cmdb_id, trace_name) pairs ranked by count_high_anomalies (tie-broken by severity_ratio).
  - Unique components present in the top results (up to 10): IG01, IG02, MG01, MG02, Tomcat01, Tomcat03, Tomcat04, dockerA1, dockerA2, dockerB2.

- Top examples / observations:
  1. MG02 / trace.from_dockerA1.row_count
     - total_points_in_window: 26, count_high_anomalies: 7
     - earliest high: 2021-03-09 01:01:00+08:00
     - max_value_in_window: 5, global_P95: 4, severity_ratio ≈ 0.25
  2. dockerA1 / trace.to_MG02.row_count (mirrors #1)
     - total_points: 26, count_high: 7, earliest: 01:01:00+08, max: 5, global_P95: 4, severity ≈ 0.25
  3. IG02 ↔ Tomcat03 row_count pairs
     - count_high_anomalies: 4, earliest ~01:02:00+08, severity ≈ 0.167
  4. Tomcat04 trace.self.duration_p95 and dockerB2 trace.self.duration_p95
     - count_high_anomalies: 4, small but measurable severity (0.128 and 0.025 respectively)
  5. Notable high-severity outlier (lower count): MG02 / trace.from_dockerB1.duration_p95
     - count_high_anomalies: 2, max_value ≈ 0.497 vs global_P95 ≈ 0.0111 → very large severity (~43.7). Investigate this duration spike even though it’s short-lived.

- Timing pattern:
  - Many earliest high-anomaly timestamps cluster around 01:01–01:02+08:00 (early in the window), with some at 01:00 or later times (e.g., Tomcat04 at 01:00).

- Next steps / recommendations:
  - Investigate the MG02 ↔ dockerA1 interaction (highest count of high anomalies).
  - Inspect the short but extreme duration spike for MG02 ← dockerB1 (trace.from_dockerB1.duration_p95) despite its low count—this may indicate intermittent latency.
  - If you want more sensitivity, consider relaxing the threshold to >= global P90 to capture additional candidates.

Stored results for follow-up (in kernel): trace_result_table (top 20 compact results) and trace_unique_cmdbs (unique cmdb_ids).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                        trace_name  total_points_in_window  count_high_anomalies earliest_high_anomaly_ts_Asia/Shanghai  max_value_in_window  global_P95  severity_ratio
0       MG02     trace.from_dockerA1.row_count                      26                     7              2021-03-09 01:01:00+08:00              5.00000    4.000000        0.250000
1   dockerA1           trace.to_MG02.row_count                      26                     7              2021-03-09 01:01:00+08:00              5.00000    4.000000        0.250000
2       IG02       trace.to_Tomcat03.row_count                      30                     4              2021-03-09 01:02:00+08:00             21.00000   18.000000        0.166667
3   Tomcat03         trace.from_IG02.row_count                      30                     4              2021-03-09 01:02:00+08:00             21.00000   18.000000        0.166667
4   Tomcat04           trace.self.duration_p95                      30                     4              2021-03-09 01:00:00+08:00              0.05300    0.047000        0.127660
5   dockerB2           trace.self.duration_p95                      30                     4              2021-03-09 01:00:00+08:00              0.01025    0.010000        0.025000
6       IG02       trace.to_Tomcat04.row_count                      30                     3              2021-03-09 01:02:00+08:00             33.00000   20.000000        0.650000
7   Tomcat04         trace.from_IG02.row_count                      30                     3              2021-03-09 01:02:00+08:00             33.00000   20.000000        0.650000
8       IG01       trace.to_Tomcat04.row_count                      30                     3              2021-03-09 01:01:00+08:00             23.00000   20.000000        0.150000
9   Tomcat04         trace.from_IG01.row_count                      30                     3              2021-03-09 01:01:00+08:00             23.00000   20.000000        0.150000
10      MG01       trace.to_dockerB2.row_count                      30                     3              2021-03-09 01:01:00+08:00             14.00000   13.000000        0.076923
11  dockerB2         trace.from_MG01.row_count                      30                     3              2021-03-09 01:01:00+08:00             14.00000   13.000000        0.076923
12      MG02       trace.to_dockerB2.row_count                      30                     3              2021-03-09 01:01:00+08:00             15.00000   14.000000        0.071429
13  dockerB2         trace.from_MG02.row_count                      30                     3              2021-03-09 01:01:00+08:00             15.00000   14.000000        0.071429
14      MG01     trace.from_Tomcat01.row_count                      30                     3              2021-03-09 01:08:00+08:00             21.00000   20.000000        0.050000
15  Tomcat01           trace.to_MG01.row_count                      30                     3              2021-03-09 01:08:00+08:00             21.00000   20.000000        0.050000
16  dockerA2              trace.self.row_count                      30                     3              2021-03-09 01:02:00+08:00            182.00000  176.000000        0.034091
17      MG01     trace.from_dockerA1.row_count                      25                     3              2021-03-09 01:05:00+08:00              6.00000    6.000000        0.000000
18  dockerA1           trace.to_MG01.row_count                      25                     3              2021-03-09 01:05:00+08:00              6.00000    6.000000        0.000000
19      MG02  trace.from_dockerB1.duration_p95                      28                     2              2021-03-09 01:00:00+08:00              0.49670    0.011115       43.687359, 0        IG01
1        IG02
2        MG01
3        MG02
4    Tomcat01
5    Tomcat03
6    Tomcat04
7    dockerA1
8    dockerA2
9    dockerB2
Name: cmdb_id, dtype: object, '')```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing in-memory log_df if present
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv", low_memory=False)

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P90 for each (cmdb_id, log_name) using full series (global thresholds BEFORE filtering)
log_thresholds = (
    log_df
    .groupby(["cmdb_id", "log_name"], dropna=False)["value"]
    .agg(global_P90=lambda x: x.quantile(0.90),
         global_P95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter log rows to the incident window inclusive (Asia/Shanghai)
# Create tz-aware local timestamp column for filtering/reporting
log_df["ts_local"] = pd.to_datetime(log_df["timestamp"].astype(int), unit="s", utc=True).dt.tz_convert(tz)

start = pd.Timestamp("2021-03-09 01:00:00", tz=tz)
end = pd.Timestamp("2021-03-09 01:30:00", tz=tz)

log_window = log_df[(log_df["ts_local"] >= start) & (log_df["ts_local"] <= end)].copy()

# 3) Merge global_P95 into the windowed data and compute per-pair metrics
log_window = log_window.merge(log_thresholds[["cmdb_id", "log_name", "global_P95"]], on=["cmdb_id", "log_name"], how="left")

def compute_log_metrics(g):
    total_points = int(g.shape[0])
    max_value = g["value"].max()
    global_p95 = g["global_P95"].iloc[0] if "global_P95" in g.columns else np.nan

    if pd.isna(global_p95):
        count_high = 0
        earliest_high_ts = pd.NaT
        severity = np.nan
    else:
        mask = g["value"] >= global_p95
        count_high = int(mask.sum())
        earliest_high_ts = g.loc[mask, "ts_local"].min() if count_high > 0 else pd.NaT
        if global_p95 == 0:
            severity = np.nan
        else:
            severity = (max_value - global_p95) / global_p95

    return pd.Series({
        "total_points_in_window": total_points,
        "count_high_anomalies": count_high,
        "earliest_high_anomaly_ts_Asia/Shanghai": earliest_high_ts,
        "max_value_in_window": max_value,
        "global_P95": global_p95,
        "severity_ratio": severity
    })

log_agg = (
    log_window
    .groupby(["cmdb_id", "log_name"], dropna=False)
    .apply(compute_log_metrics)
    .reset_index()
)

# 4) Prepare compact table: top 20 rows sorted by count_high_anomalies desc, tie-breaker severity_ratio desc
log_high = log_agg[log_agg["count_high_anomalies"] > 0].copy()

if log_high.shape[0] == 0:
    log_result_table = pd.DataFrame(columns=[
        "cmdb_id","log_name","total_points_in_window","count_high_anomalies",
        "earliest_high_anomaly_ts_Asia/Shanghai","max_value_in_window","global_P95","severity_ratio"
    ])
    log_unique_cmdbs = pd.Series([], dtype=object, name="cmdb_id")
    log_no_high_anoms_message = ("No high anomalies found in logs at >= global P95 in the incident window. "
                                "You may next inspect targeted metric KPIs (CPU/memory/disk/network) for the trace-identified candidates: "
                                "IG02, MG02, dockerA1, dockerB1, Tomcat03, Tomcat04.")
else:
    log_result_table = log_high.sort_values(by=["count_high_anomalies", "severity_ratio"], ascending=[False, False]).head(20)
    log_result_table = log_result_table[[
        "cmdb_id","log_name","total_points_in_window","count_high_anomalies",
        "earliest_high_anomaly_ts_Asia/Shanghai","max_value_in_window","global_P95","severity_ratio"
    ]].reset_index(drop=True)
    log_unique_cmdbs = pd.Series(sorted(log_result_table["cmdb_id"].dropna().unique())[:10], name="cmdb_id")
    log_no_high_anoms_message = ""

# Final compact outputs (stored in variables for subsequent steps)
log_result_table, log_unique_cmdbs, log_no_high_anoms_message

```
Out[5]:
```
Summary of log analysis (incident window 2021-03-09 01:00–01:30 Asia/Shanghai):

- What was done:
  - Global P90/P95 were computed per (cmdb_id, log_name) from the full log series, then window rows were evaluated against global P95.
  - Returned a compact top table (stored as log_result_table) and the unique cmdb_ids present (stored as log_unique_cmdbs).

- Key findings (from the compact top results):
  1. log.error_count:
     - Tomcat01/02/03/04 and apache01/02 show count_high_anomalies = 30 (every point in the window).
     - However, max_value = 0 and global_P95 = 0, so these are artifacts of a zero baseline (threshold = 0). These do NOT indicate real error spikes.
  2. log.row_count (meaningful increases):
     - Tomcat01, apache01, Tomcat04, Tomcat03, apache02, Tomcat02 each have 2 points >= global_P95 with moderate severity ratios (examples: Tomcat01 ~0.203, apache01 ~0.186, Tomcat04 ~0.147).
     - Earliest high row_count timestamps are around 01:01–01:07+08:00 (varies per component).
     - These indicate short increases in log volume/throughput worth investigating.
  3. log.gc_pause_max:
     - Tomcat03: 3 points in window equal global_P95 (max 0.08, P95 0.08 → severity 0). Tomcat02 and Tomcat01 show 1–2 elevated gc_pause points (small severity).
     - GC pause entries are present but not strongly severe except Tomcat03 has all points at the P95 level.

- Interpretation and recommendations:
  - Ignore the error_count “anomalies” where P95==0 and values==0 (not actionable).
  - Investigate components with elevated log.row_count (Tomcat01, apache01, Tomcat04, Tomcat03, apache02, Tomcat02) to see if increased request/processing load coincides with the trace anomalies.
  - Check GC pause behavior for Tomcat03 (and minor pauses on Tomcat02/01).
  - For the trace-identified candidates from earlier (IG02, MG02, dockerA1, dockerB1, Tomcat03, Tomcat04), next inspect targeted metric KPIs (CPU, memory, disk I/O, network) around the earliest anomaly times to find potential root causes.

Stored outputs in kernel for follow-up:
- log_result_table (compact top-20 results)
- log_unique_cmdbs (unique components in that top table)

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  total_points_in_window  count_high_anomalies earliest_high_anomaly_ts_Asia/Shanghai  max_value_in_window  global_P95  severity_ratio
0   Tomcat01   log.error_count                      30                    30              2021-03-09 01:00:00+08:00                 0.00      0.0000             NaN
1   Tomcat02   log.error_count                      30                    30              2021-03-09 01:00:00+08:00                 0.00      0.0000             NaN
2   Tomcat03   log.error_count                      30                    30              2021-03-09 01:00:00+08:00                 0.00      0.0000             NaN
3   Tomcat04   log.error_count                      30                    30              2021-03-09 01:00:00+08:00                 0.00      0.0000             NaN
4   apache01   log.error_count                      30                    30              2021-03-09 01:00:00+08:00                 0.00      0.0000             NaN
5   apache02   log.error_count                      30                    30              2021-03-09 01:00:00+08:00                 0.00      0.0000             NaN
6   Tomcat03  log.gc_pause_max                       3                     3              2021-03-09 01:04:00+08:00                 0.08      0.0800        0.000000
7   Tomcat01     log.row_count                      30                     2              2021-03-09 01:04:00+08:00                43.00     35.7500        0.202797
8   apache01     log.row_count                      30                     2              2021-03-09 01:07:00+08:00                84.00     70.8500        0.185603
9   Tomcat04     log.row_count                      30                     2              2021-03-09 01:02:00+08:00                48.00     41.8500        0.146953
10  Tomcat03     log.row_count                      30                     2              2021-03-09 01:01:00+08:00                45.00     41.8500        0.075269
11  apache02     log.row_count                      30                     2              2021-03-09 01:07:00+08:00                78.00     74.2000        0.051213
12  Tomcat02     log.row_count                      30                     2              2021-03-09 01:08:00+08:00                43.00     41.5500        0.034898
13  Tomcat02  log.gc_pause_max                      30                     2              2021-03-09 01:02:00+08:00                 2.08      2.0655        0.007020
14  Tomcat01  log.gc_pause_max                       3                     1              2021-03-09 01:01:00+08:00                 0.07      0.0690        0.014493
15  Tomcat04  log.gc_pause_max                       3                     1              2021-03-09 01:27:00+08:00                 0.07      0.0690        0.014493, 0    Tomcat01
1    Tomcat02
2    Tomcat03
3    Tomcat04
4    apache01
5    apache02
Name: cmdb_id, dtype: object, '')```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing 'agg' (window-stat per (cmdb_id,kpi_name)) computed earlier.
# Ensure agg exists
if 'agg' not in globals():
    raise NameError("Required variable 'agg' not found. Run previous metric window stats step first.")

# 1) Candidate components
candidates = ["IG02", "MG02", "dockerA1", "dockerB1", "Tomcat03", "Tomcat04", "Tomcat01"]

# 2) Filter agg for candidates and (count_high_anomalies>0 OR severity_ratio>0)
# Handle NaN severity_ratio by treating NaN as 0 for the selection decision
sel = agg[
    (agg["cmdb_id"].isin(candidates)) &
    (
        (agg["count_high_anomalies"].fillna(0) > 0) |
        (agg["severity_ratio"].fillna(0) > 0)
    )
].copy()

# 3) Prepare output columns
out_cols = [
    "cmdb_id",
    "kpi_name",
    "total_points_in_window",
    "count_high_anomalies",
    "earliest_high_anomaly_ts_Asia/Shanghai",
    "max_value_in_window",
    "global_P95",
    "severity_ratio"
]

# Ensure all requested columns exist in sel (some may be missing if upstream changed)
for c in out_cols:
    if c not in sel.columns:
        sel[c] = pd.NA

# 4) Sort by severity_ratio desc then count_high_anomalies desc, limit top 20
candidate_metric_anoms = sel.sort_values(by=["severity_ratio", "count_high_anomalies"], ascending=[False, False])[out_cols].reset_index(drop=True).head(20)

# 5) Short list of unique cmdb_ids present (up to 10)
unique_candidate_cmdbs = pd.Series(sorted(candidate_metric_anoms["cmdb_id"].dropna().unique())[:10], name="cmdb_id")

# 6) Components with no metric anomalies found
found_cmdbs = set(candidate_metric_anoms["cmdb_id"].dropna().unique())
no_metric_anomaly_found = [c for c in candidates if c not in found_cmdbs]

# Final compact outputs (stored in variables for inspection)
candidate_metric_anoms, unique_candidate_cmdbs, no_metric_anomaly_found

```
Out[6]:
```
summary = (
    "Metric check for candidates (IG02, MG02, dockerA1, dockerB1, Tomcat03, Tomcat04, Tomcat01):\n\n"
    "Findings:\n"
    "- Components with metric anomalies detected: IG02, MG02, Tomcat01, Tomcat03, Tomcat04.\n"
    "- Components with NO metric anomalies found in the window: dockerA1, dockerB1.\n\n"
    "Key anomalous signals (top observations):\n"
    "- Tomcat03: very high IO-wait (OSLinux-CPU_CPU_CPUWio) with earliest hit 2021-03-09 01:00+08 and extreme severity (~15x P95). Multiple disk KPIs on sdb/sda show 1.0× or higher spikes (DSKRead, DSKRTps, DSKPer..., DSKRead on sdb shows very large absolute increase). This indicates severe disk I/O pressure on Tomcat03.\n"
    "- Tomcat04: elevated IO-wait (CPUWio) starting 01:00+08 with high severity (~5.8× P95) and several disk metrics on sda/sdb showing ~0.82× or 1.0× increases. Disk I/O also a strong suspect for Tomcat04.\n"
    "- IG02: network bandwidth util on ens160 shows a jump (earliest 01:06+08) with severity ~3× P95; also local disk sda/sdb TPS/Bps shows increases at ~01:03–01:11. Both network and disk activity on IG02 merit inspection.\n"
    "- MG02: local disk throughput/read (sda DSKBps and DSKRead variants) show 0.69–1.0× severity with earliest spikes ~01:12–01:24, indicating elevated disk I/O on MG02.\n"
    "- Tomcat01: sda disk Bps/Tps spikes (earliest ~01:03–01:14) with severity ~0.82× P95 — moderate disk activity.\n\n"
    "Overall interpretation and recommended next steps:\n"
    "1) Primary symptom across affected servers is disk I/O pressure (Tomcat03, Tomcat04, MG02, Tomcat01) and IO-wait on Tomcat03/04 — investigate storage subsystem, processes causing heavy reads/writes, and any backups/cron jobs.\n"
    "2) IG02 shows a network bandwidth spike plus disk activity — check network interface (ens160) and simultaneous disk/IO patterns.\n"
    "3) dockerA1 and dockerB1 had no metric anomalies in the window — unlikely to be primary metric sources based on current thresholds.\n"
    "4) Correlate these metric spikes with the trace-level anomalies already observed (MG02/dockerA1 interactions and Tomcat03/04 traces) and with the log increases (row_count) on Tomcat nodes.\n"
    "5) If you need more sensitivity, consider lowering threshold (e.g., use P90) or include single-point but large-severity spikes.\n\n"
    "Stored outputs for follow-up: candidate_metric_anoms (compact table), unique_candidate_cmdbs, and no_metric_anomaly_found."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  total_points_in_window  count_high_anomalies earliest_high_anomaly_ts_Asia/Shanghai  max_value_in_window    global_P95  severity_ratio
0   Tomcat03                             OSLinux-CPU_CPU_CPUWio                      21                     2              2021-03-09 01:00:00+08:00             5.929600      0.361700       15.393696
1   Tomcat04                             OSLinux-CPU_CPU_CPUWio                      19                     1              2021-03-09 01:00:00+08:00             4.406200      0.650050        5.778248
2       IG02    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil                      16                     1              2021-03-09 01:06:00+08:00             0.000100      0.000025        3.000000
3       IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                      11                     1              2021-03-09 01:03:00+08:00             1.000000      0.500000        1.000000
4       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                      11                     1              2021-03-09 01:12:00+08:00             7.000000      3.500000        1.000000
5   Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                      11                     1              2021-03-09 01:02:00+08:00             0.200000      0.100000        1.000000
6   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                      11                     1              2021-03-09 01:00:00+08:00             0.100000      0.050000        1.000000
7   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                      11                     1              2021-03-09 01:00:00+08:00             7.733300      3.866650        1.000000
8   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps                      11                     1              2021-03-09 01:00:00+08:00            44.133300     22.066650        1.000000
9   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead                      11                     1              2021-03-09 01:00:00+08:00         23156.800000  11578.400000        1.000000
10      IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                      14                     1              2021-03-09 01:11:00+08:00             4.000000      2.050000        0.951220
11  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                      10                     1              2021-03-09 01:14:00+08:00             1.000000      0.550000        0.818182
12  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                      10                     1              2021-03-09 01:03:00+08:00             1.000000      0.550000        0.818182
13  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                      10                     1              2021-03-09 01:20:00+08:00             0.200000      0.110000        0.818182
14  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                      10                     1              2021-03-09 01:05:00+08:00             1.000000      0.550000        0.818182
15      IG02                             OSLinux-CPU_CPU_CPUWio                      31                     2              2021-03-09 01:01:00+08:00             0.590216      0.328300        0.797795
16  Tomcat03                    OSLinux-CPU_CPU-1_SingleCpuUtil                      10                     1              2021-03-09 01:11:00+08:00            74.268600     41.431470        0.792565
17      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                      11                     1              2021-03-09 01:24:00+08:00           117.066700     66.400000        0.763053
18      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                      11                     1              2021-03-09 01:26:00+08:00           175.000000    103.000000        0.699029
19      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                      11                     1              2021-03-09 01:24:00+08:00             1.083300      0.641650        0.688304, 0        IG02
1        MG02
2    Tomcat01
3    Tomcat03
4    Tomcat04
Name: cmdb_id, dtype: object, ['dockerA1', 'dockerB1'])```
```